# NPE and FNPE Experiments with Optimized Simulator Parameters

This notebook runs NPE and FNPE experiments using the optimized simulator parameters from the parameter estimation notebook.

**Settings:**
- Sequence length (T_seg): 3000
- Number of simulations: 10,000
- FNPE pilot length: 1500
- Parameter to infer: mu only

**Outputs:**
- PPC plots comparing posterior predictive against all 7 real trajectories
- Diffusion traces for FNPE

In [3]:
# Cell 1: Imports and Setup
import sys
import os
import json
import pickle
from pathlib import Path
from datetime import datetime

import numpy as np
import pandas as pd
import torch
import jax
import jax.numpy as jnp
import matplotlib.pyplot as plt

# Add parent directory to path
sys.path.insert(0, str(Path.cwd().parent))
os.chdir(Path.cwd().parent)

from configs.config import ExperimentConfig
from utils.env_utils import setup_environment, get_device
from utils.normalization import fit_normalizer, Normalizer
from utils.real_data import (
    build_real_window_from_csv,
    posterior_predictive_from_real,
    OBS_LABELS,
    prep_x_obs_from_df,
    make_controls_from_df,
)
from utils.plots import plot_ppc_trajectories, plot_diffusion_traces
from simulation.simulation import init_simulation_from_config, make_simulator, generate_dataset
from models.models import build_prior
from methods import build_method

# Set up environment
setup_environment()
device = get_device("cuda")  # Use CPU for stability
print(f"Using device: {device}")
print(f"JAX devices: {jax.devices()}")

Torch device: cuda | CUDA available: True CUDA version: 12.6
JAX backend: cpu
Total VRAM: 8191.50 MB | Free (driver): 7098.00 MB
Using device: cuda
JAX devices: [CpuDevice(id=0)]


In [6]:
# Cell 2: Load Optimized Parameters
import os
results_path = Path(r"c:\Users\aritr\Documents\thesis-sbi-aritra\code\experiments\advanced_parameter_estimation_results.json")
with open(results_path) as f:
    opt_results = json.load(f)

opt_params = opt_results["optimized_parameters"]
print("Optimized Parameters from Parameter Estimation:")
print("="*60)
for k, v in opt_params.items():
    default_val = opt_results["default_parameters"].get(k, "N/A")
    print(f"  {k:15s}: {v:12.6g}  (default: {default_val})")

# Key values for config
OPTIMIZED_MU = opt_params["mu"]
OPTIMIZED_CD = opt_params["cd"]
OPTIMIZED_MASS = opt_params["mass"]
print(f"\nValues for fixed parameters:")
print(f"  fixed_mu: {OPTIMIZED_MU:.4f}")
print(f"  fixed_cd: {OPTIMIZED_CD:.4f}")
print(f"  fixed_m:  {OPTIMIZED_MASS:.1f}")

Optimized Parameters from Parameter Estimation:
  mu             :     0.483798  (default: 1.0)
  cd             :     0.757142  (default: 0.27)
  mass           :      2174.57  (default: 1720.0)
  Inertia_tire   :      59.9982  (default: 28.6)
  radius_tire    :     0.353315  (default: 0.3116)
  c_1x           :  3.86507e+07  (default: 25000000.0)
  c_2x           :       255458  (default: 3600000.0)
  C_x            :       1.8913  (default: 1.42)
  E_x            :     -4.36307  (default: -9.75)
  C_roll1        :    0.0184632  (default: 0.0083)
  C_roll2        :   0.00199585  (default: 0.0005)

Values for fixed parameters:
  fixed_mu: 0.4838
  fixed_cd: 0.7571
  fixed_m:  2174.6


In [7]:
# Cell 3: Update VehicleModel Default Parameters
# This modifies the simulator to use our optimized parameters

from simulation import VehicleModel

# Store original values
original_default_params = VehicleModel.default_params.copy()
original_radius_tire = VehicleModel.radius_tire

# Update default_params with optimized values
VehicleModel.default_params["c_1x"] = opt_params["c_1x"]
VehicleModel.default_params["c_2x"] = opt_params["c_2x"]
VehicleModel.default_params["C_x"] = opt_params["C_x"]
VehicleModel.default_params["E_x"] = opt_params["E_x"]
VehicleModel.default_params["C_roll1"] = opt_params["C_roll1"]
VehicleModel.default_params["C_roll2"] = opt_params["C_roll2"]
VehicleModel.default_params["mass"] = opt_params["mass"]
VehicleModel.default_params["Inertia_tire"] = opt_params["Inertia_tire"]
VehicleModel.default_params["air_resistance"] = opt_params["cd"]

# Update radius_tire constant
VehicleModel.radius_tire = opt_params["radius_tire"]

print("✓ Updated VehicleModel with optimized parameters")
print(f"\nUpdated default_params:")
for k, v in VehicleModel.default_params.items():
    orig = original_default_params.get(k, "N/A")
    marker = " *" if v != orig else ""
    print(f"  {k:15s}: {v:12.6g}{marker}")
print(f"\n  radius_tire: {VehicleModel.radius_tire:.4f} (was {original_radius_tire:.4f})")

✓ Updated VehicleModel with optimized parameters

Updated default_params:
  c_1x           :  3.86507e+07 *
  c_2x           :       255458 *
  c_1y           :      1.9e+06
  c_2y           :       135000
  C_x            :       1.8913 *
  C_y            :          1.9
  E_x            :     -4.36307 *
  E_y            :         0.52
  C_roll1        :    0.0184632 *
  C_roll2        :   0.00199585 *
  mass           :      2174.57 *
  Inertia_z      :         2066
  Inertia_tire   :      59.9982 *
  Inertia_engine :        0.197
  air_resistance :     0.757142 *
  dt             :         0.01

  radius_tire: 0.3533 (was 0.3116)


In [9]:
# Cell 4: Load All 7 Real Trajectory CSV Files
data_dir = Path(r"c:\Users\aritr\Documents\thesis-sbi-aritra\data\measurements")
csv_files = sorted(data_dir.glob("*.csv"))

print(f"Found {len(csv_files)} CSV files:")
for i, f in enumerate(csv_files):
    df = pd.read_csv(f)
    print(f"  {i+1}. {f.name}: {len(df)} rows")

# Store dataframes
real_data_files = {}
for f in csv_files:
    name = f.stem
    real_data_files[name] = {
        'path': f,
        'df': pd.read_csv(f)
    }
    # Determine surface type
    if 'asphalt' in name.lower():
        real_data_files[name]['surface'] = 'Asphalt'
    elif 'beton' in name.lower():
        real_data_files[name]['surface'] = 'Concrete'
    elif 'basalt' in name.lower():
        real_data_files[name]['surface'] = 'Basalt'
    else:
        real_data_files[name]['surface'] = 'Unknown'

print(f"\n✓ Loaded {len(real_data_files)} trajectory files")

Found 7 CSV files:
  1. Jeversen_2021_12_15_112145_Asphalt-Vollbremsung.csv: 1070 rows
  2. Jeversen_2021_12_15_112328_Asphalt-Vollbremsung.csv: 1436 rows
  3. Jeversen_2021_12_15_125650_Beton-Vollbremsung.csv: 1189 rows
  4. Jeversen_2021_12_15_125936_Beton-Vollbremsung.csv: 1263 rows
  5. Jeversen_2021_12_15_134709_Basalt-Vollbremsung.csv: 1377 rows
  6. Jeversen_2021_12_15_134858_Basalt-Vollbremsung.csv: 1356 rows
  7. Jeversen_2022_10_12_110132.csv: 3165 rows

✓ Loaded 7 trajectory files


In [10]:
# Cell 5: Create Experiment Configuration
# Settings: T_seg=3000, num_sims=10k, only mu parameter

def create_config(method: str):
    """Create experiment config for NPE or FNPE."""
    cfg = ExperimentConfig(
        exp_name=f"optimized_{method}_mu",
        method=method,
        device="cpu",
        
        # Simulation settings
        T_seg=3000,
        dt=0.01,
        
        # Only infer mu, fix cd and m to optimized values
        active_parameters=("mu",),
        fixed_mu=OPTIMIZED_MU,  # Will be overwritten by prior during inference
        fixed_cd=OPTIMIZED_CD,
        fixed_m=OPTIMIZED_MASS,
        
        # Prior bounds for mu (adjusted based on optimization)
        prior_low_mu=0.3,
        prior_high_mu=1.0,  # Optimized mu was ~0.48, so narrow the prior
        
        # Training settings
        num_simulations=10000,
        training_batch_size=256,
        learning_rate=5e-4,
        
        # FNPE-specific
        fnpe_pilot_length=1500,
        fnpe_num_simulations=10000,
        fnpe_window_size=2,
        fnpe_score_fn_type="fnpe",  # Fast inference
        fnpe_num_diffusion_steps=500,
        fnpe_max_obs_len=50,
        
        # Encoder
        encoder_type="bigru",
        encoder_hidden=32,
        embedding_output_dim=64,
        
        # Disable unnecessary diagnostics
        run_sbc=False,
        run_lc2st=False,
        run_swd=False,
        run_one_step_rmse=False,
        run_posterior_plots=False,
        no_plots=True,  # We'll generate our own PPC plots
        
        # Output
        results_root="experiments",
    )
    return cfg

print("Configuration factory created")
print(f"\nKey settings:")
print(f"  T_seg: 3000")
print(f"  num_simulations: 10,000")
print(f"  active_parameters: ('mu',)")
print(f"  fixed_cd: {OPTIMIZED_CD:.4f}")
print(f"  fixed_m: {OPTIMIZED_MASS:.1f}")

Configuration factory created

Key settings:
  T_seg: 3000
  num_simulations: 10,000
  active_parameters: ('mu',)
  fixed_cd: 0.7571
  fixed_m: 2174.6


## NPE Experiment

In [14]:
# Cell 6: NPE - Setup, Data Generation and Normalization
print("="*70)
print("NPE EXPERIMENT")
print("="*70)

cfg_npe = create_config("npe")

# Build prior
prior_npe = build_prior(cfg_npe, device)
print(f"\nPrior: Uniform[{cfg_npe.prior_low_mu}, {cfg_npe.prior_high_mu}] for mu")

# Build simulator
print("Initializing simulator...")
init_simulation_from_config(cfg_npe)  # Initialize VehicleModel with config
simulator_npe = make_simulator(cfg_npe, device)
print(f"Simulator ready: T_seg={cfg_npe.T_seg}, dt={cfg_npe.dt}")

# Generate training data (returns theta, x, controls)
print(f"\nGenerating {cfg_npe.num_simulations:,} training simulations...")
print("This may take several minutes...")
theta_train_npe, x_train_npe, _ = generate_dataset(
    cfg_npe, prior_npe, simulator_npe
)
print(f"Training data shapes: theta={theta_train_npe.shape}, x={x_train_npe.shape}")

# Fit normalizer
normalizer_npe = fit_normalizer(theta_train_npe, x_train_npe, cfg_npe.obs_dim)

# Normalize training data
theta_train_norm_npe = normalizer_npe.normalize_theta(theta_train_npe)
x_train_norm_npe = normalizer_npe.normalize_x(x_train_npe, cfg_npe.obs_dim)
print("✓ Data normalized")

NPE EXPERIMENT

Prior: Uniform[0.3, 1.0] for mu
Initializing simulator...
Simulator ready: T_seg=3000, dt=0.01

Generating 10,000 training simulations...
This may take several minutes...
Compiling JAX (one-time JIT warmup)…


Generating sims: 100%|██████████| 10000/10000 [06:59<00:00, 23.85sims/s, 10000/10000]


Training data shapes: theta=torch.Size([10000, 1]), x=torch.Size([10000, 3000, 13])
✓ Data normalized


In [ ]:
# Cell 7: NPE - Build and Train
from sbi import utils as sbi_utils

# Build normalized prior using config bounds directly (on CPU for normalizer)
prior_low = torch.tensor([[cfg_npe.prior_low_mu]])  # (1, 1), CPU
prior_high = torch.tensor([[cfg_npe.prior_high_mu]])  # (1, 1), CPU

# Normalize and squeeze only batch dim, keep parameter dim
prior_low_norm = normalizer_npe.normalize_theta(prior_low).squeeze(0)  # (1,)
prior_high_norm = normalizer_npe.normalize_theta(prior_high).squeeze(0)  # (1,)

prior_norm_npe = sbi_utils.BoxUniform(
    low=prior_low_norm.to(device),
    high=prior_high_norm.to(device),
    device=device
)
print(f"Normalized prior bounds: low={prior_low_norm.item():.3f}, high={prior_high_norm.item():.3f}")

# Build method using method_name from config
method_npe = build_method(cfg_npe.method, cfg_npe, prior_norm_npe, device)
input_dim = x_train_norm_npe.shape[-1]
seq_len = x_train_norm_npe.shape[1]
method_npe.build(input_dim, seq_len)

# Train
print("\nTraining NPE...")
summary_npe = method_npe.train(theta_train_norm_npe, x_train_norm_npe)
print(f"\nTraining complete!")
print(f"  Final training loss: {summary_npe.get('train_loss', [None])[-1]}")
print(f"  Best validation loss: {summary_npe.get('best_val_loss', 'N/A')}")

# Build posterior
posterior_npe = method_npe.build_posterior()
print("✓ NPE posterior built")

Normalized prior bounds: low=-3.012, high=3.003

Training NPE...


c:\Users\aritr\anaconda3\envs\sbi_env\lib\site-packages\sbi\neural_nets\factory.py:287: UserWarning: The passed embedding net will be moved to cpu for
                        constructing the net building function.
  check_net_device(embedding_net, "cpu", embedding_net_warn_msg),
c:\Users\aritr\anaconda3\envs\sbi_env\lib\site-packages\sbi\inference\trainers\npe\npe_base.py:177: UserWarning: Data x has device 'cpu'. Moving x to the data_device 'cuda'. Training will proceed on device 'cuda'.
  theta, x = validate_theta_and_x(
c:\Users\aritr\anaconda3\envs\sbi_env\lib\site-packages\sbi\inference\trainers\npe\npe_base.py:177: UserWarning: Parameters theta has device 'cpu'. Moving theta to the data_device 'cuda'. Training will proceed on device 'cuda'.
  theta, x = validate_theta_and_x(
c:\Users\aritr\anaconda3\envs\sbi_env\lib\site-packages\sbi\neural_nets\net_builders\flow.py:149: UserWarning: In one-dimensional output space, this flow is limited to Gaussians
  x_numel = get_numel(


 Training neural network. Epochs trained: 34

## FNPE Experiment

In [ ]:
# Cell 8: FNPE - Setup and Training
print("="*70)
print("FNPE EXPERIMENT")
print("="*70)

cfg_fnpe = create_config("fnpe")

# For FNPE, we need to use the physical prior
prior_fnpe = build_prior(cfg_fnpe, device)

# Build FNPE method (it has its own data generation)
method_fnpe = build_method(cfg_fnpe, prior_fnpe, device)

# Build method
input_dim_fnpe = cfg_fnpe.obs_dim + 2  # obs + controls (steer, accel)
method_fnpe.build(input_dim_fnpe, cfg_fnpe.T_seg)

print("\nTraining FNPE...")
print(f"  Pilot length: {cfg_fnpe.fnpe_pilot_length}")
print(f"  Budget: {cfg_fnpe.fnpe_num_simulations}")
print(f"  Window size: {cfg_fnpe.fnpe_window_size}")

# Train FNPE (it generates its own data internally)
summary_fnpe = method_fnpe.train(
    num_simulations=cfg_fnpe.fnpe_num_simulations,
    T_obs=cfg_fnpe.T_seg
)

print(f"\nFNPE Training complete!")

In [ ]:
# Cell 9: FNPE - Build Posterior
# Build FNPE posterior with normalizer for consistent interface
from methods.fnpe_method import FNPEPosterior

# Create normalizer for FNPE (using same stats as NPE for consistency)
normalizer_fnpe = normalizer_npe  # Reuse NPE normalizer

# Build FNPE posterior wrapper
posterior_fnpe = method_fnpe.build_posterior(normalizer=normalizer_fnpe)
print("✓ FNPE posterior built")

## PPC Plots for All 7 Trajectories

In [ ]:
# Cell 10: Helper Function for PPC
def run_ppc_for_trajectory(
    traj_name: str,
    df: pd.DataFrame,
    cfg: ExperimentConfig,
    posterior,
    normalizer: Normalizer,
    method_name: str,
    K_ppc: int = 100,
    start_offset: int = 50,
    T_seg: int = 900,  # Use shorter segment for PPC
):
    """Run PPC for a single trajectory."""
    try:
        # Build real window
        x_obs_full, controls_real, start_idx = build_real_window_from_csv(
            df,
            cfg,
            device,
            start_idx=start_offset,
            prefer_low_brake=False,
            rate_body_z_in_deg_s=True,
            tire_rates_in_rpm=False,
            vel_body_in_kmh=False,
        )
        
        # Truncate to T_seg if needed
        T_actual = min(T_seg, x_obs_full.shape[1])
        x_obs_trunc = x_obs_full[:, :T_actual, :]
        controls_trunc = {
            k: v[:T_actual] if hasattr(v, '__len__') and len(v) > T_actual else v
            for k, v in controls_real.items()
        }
        
        # Run PPC
        y_real, y_ppc = posterior_predictive_from_real(
            posterior,
            x_obs_trunc,
            controls_trunc,
            cfg,
            normalizer=normalizer,
            device=device,
            K_ppc=K_ppc,
        )
        
        return y_real, y_ppc, True
    except Exception as e:
        print(f"  Error: {e}")
        return None, None, False

print("✓ PPC helper function defined")

In [ ]:
# Cell 11: Run PPC for NPE on All 7 Trajectories
print("="*70)
print("NPE - Posterior Predictive Checks on All 7 Real Trajectories")
print("="*70)

# Create output directory
fig_dir = Path("experiments/optimized_params_ppc")
fig_dir.mkdir(parents=True, exist_ok=True)

K_PPC = 100  # Number of posterior samples
T_PPC = 900  # Segment length for PPC

npe_results = {}
for i, (name, data) in enumerate(real_data_files.items()):
    print(f"\n[{i+1}/7] {name} ({data['surface']})")
    
    y_real, y_ppc, success = run_ppc_for_trajectory(
        name, data['df'], cfg_npe, posterior_npe, normalizer_npe,
        "NPE", K_ppc=K_PPC, T_seg=T_PPC
    )
    
    if success:
        npe_results[name] = {'y_real': y_real, 'y_ppc': y_ppc, 'surface': data['surface']}
        
        # Plot
        short_name = name[:30] + "..." if len(name) > 30 else name
        out_path = fig_dir / f"ppc_npe_{i+1}_{data['surface'].lower()}.png"
        plot_ppc_trajectories(
            y_real=y_real,
            y_ppc=y_ppc,
            obs_labels=OBS_LABELS,
            dt=cfg_npe.dt,
            out_path=out_path,
            max_trajs=50,
            plot_all_trajs=True,
            max_dims=cfg_npe.obs_dim,
            title=f"NPE PPC: {data['surface']} (Traj {i+1})",
        )
        print(f"  ✓ Saved: {out_path.name}")

print(f"\n✓ NPE PPC complete for {len(npe_results)}/7 trajectories")

In [ ]:
# Cell 12: Run PPC for FNPE on All 7 Trajectories
print("="*70)
print("FNPE - Posterior Predictive Checks on All 7 Real Trajectories")
print("="*70)

fnpe_results = {}
for i, (name, data) in enumerate(real_data_files.items()):
    print(f"\n[{i+1}/7] {name} ({data['surface']})")
    
    y_real, y_ppc, success = run_ppc_for_trajectory(
        name, data['df'], cfg_fnpe, posterior_fnpe, normalizer_fnpe,
        "FNPE", K_ppc=K_PPC, T_seg=T_PPC
    )
    
    if success:
        fnpe_results[name] = {'y_real': y_real, 'y_ppc': y_ppc, 'surface': data['surface']}
        
        # Plot
        out_path = fig_dir / f"ppc_fnpe_{i+1}_{data['surface'].lower()}.png"
        plot_ppc_trajectories(
            y_real=y_real,
            y_ppc=y_ppc,
            obs_labels=OBS_LABELS,
            dt=cfg_fnpe.dt,
            out_path=out_path,
            max_trajs=50,
            plot_all_trajs=True,
            max_dims=cfg_fnpe.obs_dim,
            title=f"FNPE PPC: {data['surface']} (Traj {i+1})",
        )
        print(f"  ✓ Saved: {out_path.name}")

print(f"\n✓ FNPE PPC complete for {len(fnpe_results)}/7 trajectories")

## Diffusion Traces for FNPE

In [ ]:
# Cell 13: Generate Diffusion Traces for FNPE
print("="*70)
print("FNPE - Diffusion Traces")
print("="*70)

# Pick one trajectory for trace visualization
trace_traj_name = list(real_data_files.keys())[0]  # First trajectory
trace_data = real_data_files[trace_traj_name]

print(f"\nGenerating diffusion traces for: {trace_traj_name}")

# Build real observation window
try:
    x_obs_full, controls_real, start_idx = build_real_window_from_csv(
        trace_data['df'],
        cfg_fnpe,
        device,
        start_idx=50,
        prefer_low_brake=False,
        rate_body_z_in_deg_s=True,
        tire_rates_in_rpm=False,
        vel_body_in_kmh=False,
    )
    
    # Truncate for visualization
    T_trace = min(500, x_obs_full.shape[1])
    x_obs_trace = x_obs_full[:, :T_trace, :]
    
    # Normalize observation
    x_norm = normalizer_fnpe.normalize_x(x_obs_trace, cfg_fnpe.obs_dim)
    
    # Sample with traces
    num_trace_samples = 50
    print(f"Sampling {num_trace_samples} traces...")
    
    # Check if posterior supports sample_with_traces
    if hasattr(posterior_fnpe, 'sample_with_traces'):
        traces = posterior_fnpe.sample_with_traces(
            (num_trace_samples,),
            x=x_norm,
            return_physical=True
        )
        
        print(f"Trace shape: {traces.shape}")
        
        # Plot diffusion traces
        trace_path = fig_dir / "fnpe_diffusion_traces.png"
        plot_diffusion_traces(
            traces=traces,
            out_path=trace_path,
            theta_true=None,  # No true value for real data
            param_names=["mu"],
            title="FNPE Diffusion Traces (Real Data)",
            max_traces=50,
            alpha=0.2,
        )
        print(f"✓ Saved diffusion traces to: {trace_path}")
    else:
        print("Warning: Posterior does not support sample_with_traces")
        
except Exception as e:
    print(f"Error generating traces: {e}")
    import traceback
    traceback.print_exc()

In [ ]:
# Cell 14: Summary Comparison Plot
print("="*70)
print("Summary: NPE vs FNPE PPC Comparison")
print("="*70)

# Compute MSE for each method and trajectory
def compute_ppc_mse(y_real, y_ppc):
    """Compute MSE between real and PPC median."""
    y_median = np.median(y_ppc, axis=0)
    return np.mean((y_real - y_median) ** 2)

print("\nPPC MSE Comparison:")
print(f"{'Trajectory':<45} {'Surface':<10} {'NPE MSE':<12} {'FNPE MSE':<12}")
print("-"*80)

npe_mses = []
fnpe_mses = []

for name in real_data_files.keys():
    surface = real_data_files[name]['surface']
    short_name = name[:42] + "..." if len(name) > 42 else name
    
    npe_mse = compute_ppc_mse(npe_results[name]['y_real'], npe_results[name]['y_ppc']) if name in npe_results else float('nan')
    fnpe_mse = compute_ppc_mse(fnpe_results[name]['y_real'], fnpe_results[name]['y_ppc']) if name in fnpe_results else float('nan')
    
    if not np.isnan(npe_mse):
        npe_mses.append(npe_mse)
    if not np.isnan(fnpe_mse):
        fnpe_mses.append(fnpe_mse)
    
    print(f"{short_name:<45} {surface:<10} {npe_mse:<12.4f} {fnpe_mse:<12.4f}")

print("-"*80)
print(f"{'AVERAGE':<45} {'':<10} {np.mean(npe_mses):<12.4f} {np.mean(fnpe_mses):<12.4f}")

# Bar plot comparison
fig, ax = plt.subplots(figsize=(12, 5))

x = np.arange(len(real_data_files))
width = 0.35

npe_vals = [compute_ppc_mse(npe_results[n]['y_real'], npe_results[n]['y_ppc']) if n in npe_results else 0 for n in real_data_files.keys()]
fnpe_vals = [compute_ppc_mse(fnpe_results[n]['y_real'], fnpe_results[n]['y_ppc']) if n in fnpe_results else 0 for n in real_data_files.keys()]

ax.bar(x - width/2, npe_vals, width, label='NPE', color='C0', alpha=0.8)
ax.bar(x + width/2, fnpe_vals, width, label='FNPE', color='C1', alpha=0.8)

ax.set_ylabel('PPC MSE')
ax.set_xlabel('Trajectory')
ax.set_title('NPE vs FNPE: Posterior Predictive Check MSE\n(With Optimized Simulator Parameters, mu only)')
ax.set_xticks(x)
ax.set_xticklabels([f"{real_data_files[n]['surface']}_{i+1}" for i, n in enumerate(real_data_files.keys())], rotation=45, ha='right')
ax.legend()
ax.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
summary_path = fig_dir / "npe_vs_fnpe_ppc_comparison.png"
plt.savefig(summary_path, dpi=150)
plt.show()
print(f"\n✓ Saved comparison plot to: {summary_path}")

In [ ]:
# Cell 15: Save Results
print("="*70)
print("Saving Results")
print("="*70)

results_summary = {
    "timestamp": datetime.now().isoformat(),
    "optimized_params_used": opt_params,
    "experiment_settings": {
        "T_seg": 3000,
        "num_simulations": 10000,
        "fnpe_pilot_length": 1500,
        "active_parameters": ["mu"],
        "fixed_cd": OPTIMIZED_CD,
        "fixed_m": OPTIMIZED_MASS,
    },
    "npe_ppc_mse": {name: float(compute_ppc_mse(npe_results[name]['y_real'], npe_results[name]['y_ppc'])) 
                    for name in npe_results.keys()},
    "fnpe_ppc_mse": {name: float(compute_ppc_mse(fnpe_results[name]['y_real'], fnpe_results[name]['y_ppc'])) 
                     for name in fnpe_results.keys()},
    "average_npe_mse": float(np.mean(npe_mses)),
    "average_fnpe_mse": float(np.mean(fnpe_mses)),
}

results_path = fig_dir / "experiment_results.json"
with open(results_path, 'w') as f:
    json.dump(results_summary, f, indent=2)

print(f"✓ Results saved to: {results_path}")
print(f"\nFigures saved in: {fig_dir}")
print(f"  - PPC plots for NPE: ppc_npe_*.png")
print(f"  - PPC plots for FNPE: ppc_fnpe_*.png")
print(f"  - Diffusion traces: fnpe_diffusion_traces.png")
print(f"  - Comparison: npe_vs_fnpe_ppc_comparison.png")

In [ ]:
# Cell 16: Restore Original Parameters (cleanup)
print("\nRestoring original VehicleModel parameters...")
VehicleModel.default_params = original_default_params
VehicleModel.radius_tire = original_radius_tire
print("✓ Original parameters restored")